In [0]:
from pyspark.sql import functions as F

print("Ingesting raw batch records into Bronze Storage...")

# Product Schema Definitions
products_data = [(1, "Ergonomic Wireless Mouse", 15), (2, "27-Inch 4K Monitor", 8),
                 (3, "Mechanical Keyboard", 12), (5, "USB-C Hub", 5)]
df_products = spark.createDataFrame(products_data, ["product_id", "product_name", "reorder_threshold"])

Ingesting raw batch records into Bronze Storage...


In [0]:
# Stock Movements Transaction Feed
stock_movements_data = [(1001, 1, 1, 50), (1002, 1, 1, -40), (1003, 2, 2, 20),
                        (1004, 2, 2, -5), (1005, 3, 1, 5), (1006, 5, 2, 30),
                        (1007, 5, 2, -28), (1008, 1, 2, 100), (1009, 1, 2, -95)]
df_movements = spark.createDataFrame(stock_movements_data, ["movement_id", "product_id", "warehouse_id", "quantity_changed"])

In [0]:
warehouses_data = [(1, "Main Hub Chennai"), (2, "North Distribution Bangalore")]
df_warehouses = spark.createDataFrame(warehouses_data, ["warehouse_id", "warehouse_name"])

In [0]:
# 2. SILVER LAYER: Transformed and Enriched Core Master Tables
# -------------------------------------------------------------------------
print("Processing Silver Master Layer Tables...")

# Multi-table join to synthesize your inventory views
df_silver_inventory = df_movements \
    .join(df_products, "product_id", "inner") \
    .join(df_warehouses, "warehouse_id", "inner") \
    .withColumn("processed_timestamp", F.current_timestamp())

# Write permanently to cloud storage as a Silver Delta Table
df_silver_inventory.write.format("delta").mode("overwrite").saveAsTable("silver_inventory_movements")

Processing Silver Master Layer Tables...


In [0]:
# 3. GOLD LAYER: Business-Ready Warehouse KPI Metrics
# -------------------------------------------------------------------------
print("Aggregating Master Gold KPI Presentation Tables...")

df_gold_summary = spark.table("silver_inventory_movements") \
    .groupBy("warehouse_name", "product_name", "reorder_threshold").agg(
        F.sum("quantity_changed").alias("Current_Stock")
    )

# Append Business Rules Flags to highlight stock deficits
df_gold_final = df_gold_summary.withColumn(
    "Stock_Status",
    F.when(F.col("Current_Stock") <= F.col("reorder_threshold"), "UNDERSTOCK / REORDER")
     .when(F.col("Current_Stock") >= 50, "OVERSTOCKED ALERT")
     .otherwise("HEALTHY")
)

Aggregating Master Gold KPI Presentation Tables...


In [0]:
df_gold_final.write.format("delta").mode("overwrite").saveAsTable("gold_warehouse_stock_status")
print("Cloud Inventory Medallion Tables successfully written to Delta Lake!")

Cloud Inventory Medallion Tables successfully written to Delta Lake!


In [0]:
%sql
-- Compact storage partitions for fast indexing access
OPTIMIZE gold_warehouse_stock_status ZORDER BY (warehouse_name);

-- Check Delta ACID logging audit parameters
DESCRIBE HISTORY gold_warehouse_stock_status;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-21T09:33:32.000Z,147985412235563,azuser7213_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2509122997833647),5130f81d-e4a4-418d-96a3-42e652ba23e8,0621-083810-wzt9fomd-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2056)",null,Databricks-Runtime/18.2.x-photon-scala2.13
